In [1]:
import os
os.chdir('/home/asudupe/Latxa-Omni/')

In [2]:
import torch
import torchaudio
from omni_speech.constants import SPEECH_TOKEN_INDEX, DEFAULT_SPEECH_TOKEN
from omni_speech.conversation import conv_templates, SeparatorStyle
from omni_speech.model.builder import load_pretrained_model
from omni_speech.datasets.preprocess import tokenizer_speech_token
from torch.utils.data import Dataset, DataLoader
import whisper
from datasets import load_dataset, load_from_disk
import numpy as np
from IPython.display import Audio
from scipy.io.wavfile import write
from torchaudio.transforms import Resample
from speechbrain.inference.vocoders import UnitHIFIGAN

In [3]:
def ctc_postprocess(tokens, blank):
    _toks = tokens.squeeze(0).tolist()
    deduplicated_toks = [v for i, v in enumerate(_toks) if i == 0 or v != _toks[i - 1]]
    hyp = [v for v in deduplicated_toks if v != blank] #官方493 222
    hyp = " ".join(list(map(str, hyp))) #1918 547
    return hyp

In [8]:
dataset = load_from_disk('/scratch/asudupe/datasets/VoiceAssistant-400K_eu/dataset_with_token_paths/')

In [ ]:
speech, sr = torchaudio.load(os.path.join('/scratch/asudupe/datasets/VoiceAssistant-400K_eu',dataset['train'][10]['question_audio']))
Audio(data=np.array(speech), rate=sr)

In [15]:
dataset['train'][10]['answer']

'Txakur bat apaintzeak hainbat urrats dakartza. Has zaitez zure txakurraren ilea eskuilatzen, korapiloak eta ile solteak kentzeko. Ondoren, eman bainu bat zure txakurrari, txakurren ile-apainketarako xanpu egokia erabiliz, eta ziurtatu ondo garbitzen duzula hondakinik ez uzteko. Bainuaren ondoren, lehortu zure txakurra eskuoihal batekin edo animalientzako lehorgailu batekin. Moztu zure txakurraren azazkalak kontu handiz, azazkalaren erroa ukitu gabe. Azkenik, garbitu zure txakurraren belarriak albaitariak gomendatutako belarri-garbitzaile batekin eta garbitu hortzak txakurrentzako hortzetako pastarekin. Izan beti leuna eta eskaini sariak prozesuan zehar zure txakurra lasai eta eroso mantentzeko.'

In [6]:
write(filename='example.wav', data=np.array(dataset['train'][4]['question_audio'], dtype=np.float32), rate=22050)

In [13]:
speech_file = "omni_speech/serve/examples/helpful_base_1.wav"
speech = whisper.load_audio(speech_file)

Audio(data=np.array(speech), rate=16000)


In [4]:
# model_path = 'saves/13834/checkpoint-24000'
# model_path = "/scratch/asudupe/checkpoints/Latxa-Llama-3.1-8B-Instruct/stage1/best/checkpoint-20976"
model_path = "/scratch/asudupe/checkpoints/Latxa-Llama-3.1-8B-Instruct/stage2/3844320/checkpoint-13983"
# model_path = "Llama-3.1-8B-Omni"
model_base = None
is_lora = False
s2s = True
mel_size = 128
conv_mode = 'llama_3'

In [5]:
tokenizer, model, context_len = load_pretrained_model(model_path, model_base, is_lora=is_lora, s2s=s2s)


/scratch/asudupe/conda-env/latxa-omni/lib/python3.10/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [6]:
hifigan = UnitHIFIGAN.from_hparams(source="/scratch/asudupe/models/hifigan/gaitu/", run_opts={"device":'cuda'})

/scratch/asudupe/conda-env/latxa-omni/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:30: UserWarning: torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.
  warnings.warn("torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.")


In [9]:
speech_file = os.path.join('/scratch/asudupe/datasets/VoiceAssistant-400K_eu',dataset['test'][1010]['question_audio'])
# speech = speech.squeeze()
# speech_file = "audioak/Recording_9.mp3"
speech_loaded = whisper.load_audio(speech_file)
# speech = Resample(orig_freq=sr, new_freq=16000)(speech)
Audio(data=np.array(speech_loaded), rate = 16000)

In [10]:
qs = "<speech>\nPlease directly answer the questions in the user's speech."
# audio = dataset['train'][20]['question_audio']
# speech = torch.tensor(audio, dtype=torch.float32)
# speech = Resample(orig_freq=22050, new_freq=16000)(speech)

conv = conv_templates[conv_mode].copy()
conv.append_message(conv.roles[0], qs)
conv.append_message(conv.roles[1], None)
prompt = conv.get_prompt()

speech = whisper.pad_or_trim(speech_loaded)
speech = whisper.log_mel_spectrogram(speech, n_mels=mel_size).permute(1, 0)
input_ids = tokenizer_speech_token(prompt, tokenizer, return_tensors='pt')
speech_length = torch.LongTensor([speech.shape[0]])

input_ids = input_ids.to(device='cuda', non_blocking=True)
speech_tensor = speech.to(dtype=torch.float16, device='cuda', non_blocking=True)
speech_length = speech_length.to(device='cuda', non_blocking=True)

input_ids = input_ids.unsqueeze(0)
speech_tensors = speech_tensor.unsqueeze(0)
speech_lengths = speech_length.unsqueeze(0)

# input_ids = torch.stack((input_ids), dim=0)
# speech_tensors = torch.stack((speech_tensor), dim=0)
# speech_lengths = torch.stack((speech_length), dim=0)

#torch.Size([1, 62]),torch.Size([1, 3000, 128]) #tensor([[3000]])

In [11]:
input_ids.shape, speech_tensors.shape, speech_lengths

(torch.Size([1, 64]),
 torch.Size([1, 3000, 128]),
 tensor([[3000]], device='cuda:0'))

In [12]:
temperature = 0.0
top_p = None
num_beams = 1
max_new_tokens = 256

with torch.inference_mode():
    outputs = model.generate(
        input_ids,
        speech=speech_tensors,
        speech_lengths=speech_lengths,
        do_sample=True if temperature > 0 else False,
        temperature=temperature,
        top_p=top_p,
        num_beams=num_beams,
        max_new_tokens=max_new_tokens,
        use_cache=True,
        pad_token_id=128004,
        streaming_unit_gen=True,
 
    )
output_ids = outputs
output_ids, output_units = outputs

print(tokenizer.batch_decode(output_ids, skip_special_tokens=True)[0].strip())
output_units = ctc_postprocess(output_units, blank=model.config.unit_vocab_size)
output_units = torch.tensor([int(x) for x in output_units.split()], dtype=torch.long)


/scratch/asudupe/conda-env/latxa-omni/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:601: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)
The attention layers in this model are transitioning from computing the RoPE embeddings internally through `position_ids` (2D tensor with the indexes of the tokens), to using externally computed `position_embeddings` (Tuple of tensors, containing cos and sin). In v4.46 `position_ids` will be removed and `position_embeddings` will be mandatory.


Adimen artifizialak onura ekar diezaioke osasun-laguntzari, diagnostikoaren zehaztasuna hobetuz irudi aurreratuen eta eredu-ezagutzaren bidez. Pazienteen datu kopuru handiak azkar azter ditzake, giza begiek antzeman ditzaketen ereduak identifikatzeko, eta horrek gaixotasunen detekzio goiztiarra eta tratamendu-plan pertsonalizatuak ekar ditzake. IAk administrazio-zereginak ere arindu ditzake, hala nola programazioa eta pazienteen erregistroen kudeaketa, osasun-profesionalei pazienteen arretan gehiago zentratzeko aukera emanez. Gainera, IAk elikatutako analitika prediktiboak gaixotasun-agerraldiak iragar ditzake, prebentzio- eta erantzun-estrategiak hobetzen lagunduz. Oro har, IAk eraginkortasuna, zehaz


In [107]:
answer = hifigan.decode_unit(output_units.unsqueeze(-1), torch.tensor(np.load('/scratch/asudupe/models/hifigan/gaitu/jon.npy')))

In [15]:
Audio(answer.cpu(), rate=16000)

In [17]:
speech_file = os.path.join('/scratch/asudupe/datasets/VoiceAssistant-400K_eu',dataset['test'][1010]['answer_audio'])
speech_loaded = whisper.load_audio(speech_file)

Audio(speech_loaded, rate=16000)

In [112]:
write(filename='aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa.wav', data=answer.cpu(), rate=16000)

AttributeError: 'torch.dtype' object has no attribute 'kind'

In [116]:
torchaudio.save("audioak/aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa.wav", answer.cpu(), sample_rate=16000)